<center>
<img src="https://www.infnet.edu.br/infnet/wp-content/uploads/sites/18/2021/10/infnet-30-horizontal-padrao@300x-8-1024x265.png" width="60%"/>
</center>

# MBA em Engenharia de Dados: Big Data e IA
## Processamento de Big Data com Apache Spark e Spark SQL [26E3_2]
### Projeto da disciplina

### Variáveis dos volumes paths

In [0]:
CATALOG                     = 'instacart'

SILVER_AISLES               = f'{CATALOG}.silver.aisle'
SILVER_DEPARTMENTS          = f'{CATALOG}.silver.department'
SILVER_ORDERS               = f'{CATALOG}.silver.order'
SILVER_PRODUCTS             = f'{CATALOG}.silver.product'
SILVER_ORDER_PRODUCTS       = f'{CATALOG}.silver.order_product'

GOLD_AISLES                 = f'{CATALOG}.gold.aisle'
GOLD_DEPARTMENTS            = f'{CATALOG}.gold.department'
GOLD_ORDERS                 = f'{CATALOG}.gold.order'
GOLD_PRODUCTS               = f'{CATALOG}.gold.product'

## CAMADA GOLD

In [0]:
%sql

CREATE DATABASE IF NOT EXISTS instacart.gold;


CREATE TABLE IF NOT EXISTS instacart.gold.aisle (  
    aisle_id INTEGER,
    description STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.gold.department (  
    department_id INTEGER,
    description STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.gold.order (  
    order_id INTEGER,
    user_id INTEGER,
    product_id INTEGER,
    order_number INTEGER,
    order_day_of_week INTEGER
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.gold.product (  
    product_id INTEGER,
    description STRING,
    aisle_id INTEGER,
    department_id INTEGER
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
aisles_df = spark.table(SILVER_AISLES)
aisles_df.display()

In [0]:
departments_df = spark.table(SILVER_DEPARTMENTS)
departments_df.display()

In [0]:
products_df = spark.table(SILVER_PRODUCTS)
products_df.display()

In [0]:
orders_df = spark.table(SILVER_ORDERS)
orders_df.display()

In [0]:
orders_products_df = spark.table(SILVER_ORDER_PRODUCTS)
orders_products_df.display()

In [0]:
orders_df = (orders_df
 .join(orders_products_df.drop("eval_set"), on="order_id", how="inner")
 .select("order_id", "user_id", "product_id", "order_number", "order_day_of_week")
 .sort("order_id", "product_id")
)
orders_df.display()

In [0]:
merge_data(df=aisles_df, layer_path=GOLD_AISLES, merge_keys=['aisle_id'])
merge_data(df=departments_df, layer_path=GOLD_DEPARTMENTS, merge_keys=['department_id'])
merge_data(df=products_df, layer_path=GOLD_PRODUCTS, merge_keys=['product_id'])
merge_data(df=orders_df, layer_path=GOLD_ORDERS, merge_keys=orders_df.columns)


## ANALYTICS

A partir da camada Gold finalizada, levantou-se algumas perguntas analíticas para usufruir dos dados disponíveis nessa camada.

In [0]:
from pyspark.sql.functions import col, count, countDistinct, desc, avg, round as spark_round

In [0]:
products_df = spark.table(GOLD_PRODUCTS).alias("p")
aisles_df = spark.table(GOLD_AISLES).alias("a")
departments_df = spark.table(GOLD_DEPARTMENTS).alias("d")

### Pergunta 1: Quais são os 10 produtos mais vendidos (maior número de pedidos), e a quais corredor e departamento eles pertencem?

In [0]:
top_products_df = (
    spark.table(GOLD_ORDERS).alias("o")
    .join(products_df, col("o.product_id") == col("p.product_id"))
    .join(aisles_df, col("p.aisle_id") == col("a.aisle_id"))
    .join(departments_df, col("p.department_id") == col("d.department_id"))
    .groupBy(
        col("p.product_id"),
        col("p.description").alias("product"),
        col("a.description").alias("aisle"),
        col("d.description").alias("department"),
    )
    .agg(countDistinct("o.order_id").alias("total_orders"))
    .orderBy(desc("total_orders"))
    .limit(10)
)
top_products_df.display()

### Pergunta 2: Qual departamento concentra o maior número de itens comprados?

In [0]:
items_by_department_df = (
    spark.table(GOLD_ORDERS).alias("o")
    .join(products_df, col("o.product_id") == col("p.product_id"))
    .join(departments_df, col("p.department_id") == col("d.department_id"))
    .groupBy(col("d.description").alias("department"))
    .agg(count("*").alias("total_items_sold"))
    .orderBy(desc("total_items_sold"))
)
items_by_department_df.display()

### Pergunta 3: Em qual dia da semana ocorre o maior número de pedidos?

In [0]:
orders_by_weekday_df = (
    spark.table(GOLD_ORDERS)
    .groupBy("order_day_of_week")
    .agg(countDistinct("order_id").alias("total_orders"))
    .orderBy(desc("total_orders"))
)
orders_by_weekday_df.display()

### Pergunta 4: Qual é a média de itens (produtos) por pedido?

In [0]:
avg_items_per_order_df = (
    spark.table(GOLD_ORDERS)
    .groupBy("order_id")
    .agg(count("product_id").alias("items_per_order"))
    .agg(spark_round(avg("items_per_order"), 2).alias("avg_items_per_order"))
)
avg_items_per_order_df.display()